In [32]:
import pandas as pd
from datetime import datetime, timedelta

In [33]:
# Function to map timestamp to real date
def map_timestamp_to_date(timestamp, start_date=datetime(2022, 1, 1)):
    """
    Map numerical timestamp to a real date
    Assumes timestamp ranges from 0 to 200
    """
    # Calcola il numero di giorni da aggiungere
    days_to_add = timestamp
    
    return start_date + timedelta(days=days_to_add)

In [34]:
# Read the CSV files
accounts_df = pd.read_csv('datasets/AMLSimData/1_1K/accounts.csv')
transactions_df = pd.read_csv('datasets/AMLSimData/1_1K/transactions.csv')

# Prepare the edge list for Cosmograph
cosmograph_edges = transactions_df[['SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_AMOUNT', 'IS_FRAUD', 'TIMESTAMP']].copy()

# Rename columns to match Cosmograph requirements
cosmograph_edges.columns = ['source', 'target', 'weight', 'is_fraud', 'time']

# Convert numerical timestamp to real dates
cosmograph_edges['time'] = cosmograph_edges['time'].apply(map_timestamp_to_date)

cosmograph_edges['color'] = cosmograph_edges['is_fraud'].map({True: 'red', False: 'grey'})

# Ensure columns are in the right order
cosmograph_edges = cosmograph_edges[['source', 'target', 'weight', 'color', 'time']]

# Save the edge list for Cosmograph
cosmograph_edges.to_csv('graphs/cosmograph_edges.csv', index=False)

# Print some basic statistics
print("Total number of edges:", len(cosmograph_edges))
print("Timespan of transactions:", 
      cosmograph_edges['time'].min(), 
      "to", 
      cosmograph_edges['time'].max())
print("Fraud transactions:", cosmograph_edges['color'].value_counts())


Total number of edges: 117533
Timespan of transactions: 2022-01-01 00:00:00 to 2022-07-19 00:00:00
Fraud transactions: color
grey    117358
red        175
Name: count, dtype: int64


Remove timestamp

In [35]:
with_timestamp = pd.read_csv('cosmograph_edges.csv')

new_graph = with_timestamp[['source', 'target', 'weight', 'color']].copy()

# Rename columns to match Cosmograph requirements
new_graph.columns = ['source', 'target', 'weight', 'color']

# Save the edge list for Cosmograph
new_graph.to_csv('graphs/notime.csv', index=False)

# Print some basic statistics
print("Total number of edges:", len(new_graph))
print("Fraud transactions:", new_graph['color'].value_counts())


Total number of edges: 117533
Fraud transactions: color
grey    117358
red        175
Name: count, dtype: int64


In [36]:
# Prepare the edges
cosmograph_edges = transactions_df[['SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_AMOUNT', 'IS_FRAUD']].copy()
cosmograph_edges.columns = ['source', 'target', 'weight', 'is_fraud']


# Identify fraudulent accounts (accounts that sent fraudulent transactions)
fraudulent_accounts = set(
    cosmograph_edges[cosmograph_edges['is_fraud'] == True]['source'].unique()
)

# Prepare unique nodes
unique_nodes = pd.concat([cosmograph_edges['source'], cosmograph_edges['target']]).unique()

# Create metadata file
metadata_df = pd.DataFrame({
    'id': unique_nodes,
    'color': ['red' if node in fraudulent_accounts else 'gray' for node in unique_nodes]
})

# Save metadata file with semicolon separator
metadata_df.to_csv('graphs/metadata.csv', index=False, sep=';')

# Prepare edges file
cosmograph_edges = cosmograph_edges[
    (cosmograph_edges['source'] != cosmograph_edges['target']) & 
    (cosmograph_edges['weight'] > 0)
]

# Add edge color based on fraud status
cosmograph_edges['color'] = cosmograph_edges['is_fraud'].map({True: 'red', False: 'gray'})

# Select and rename final columns
edges_columns = ['source', 'target', 'weight', 'color']
cosmograph_edges = cosmograph_edges[edges_columns]

# Save edges file
cosmograph_edges.to_csv('graphs/graph.csv', index=False)

# Print statistics
print("Metadata file (metadata.csv):")
print(metadata_df['color'].value_counts())
print("\nTotal unique nodes:", len(metadata_df))
print("\nEdges file (graph.csv):")
print("Total edges:", len(cosmograph_edges))
print("Fraudulent edges:", len(cosmograph_edges[cosmograph_edges['color'] == 'red']))

Metadata file (metadata.csv):
color
gray    854
red     146
Name: count, dtype: int64

Total unique nodes: 1000

Edges file (graph.csv):
Total edges: 117412
Fraudulent edges: 175
